In [1]:
# check if the elasticsearch container is running

from elasticsearch import Elasticsearch

# !curl http://localhost:9200/

In [2]:
# instantiate Elasticsearch client localhost
client = Elasticsearch("http://localhost:9200")

In [3]:
# Check paths
import os
# print(os.getcwd())

# print(os.path.exists("../data/documents.csv"))

In [4]:
# Define paths

from pathlib import Path

BASE_DIR = Path().resolve().parent  # project
doc_csv_path = BASE_DIR / "data" / "documents.csv"
doc_jsonl_path = BASE_DIR / "data" / "documents.jsonl"

queries_csv_path = BASE_DIR / "data" / "queries.csv"
queries_jsonl_path = BASE_DIR / "data" / "queries.jsonl"


In [ ]:
# Function to convert csv file to jsonl

import csv
import json

def csv_to_jsonl(csv_path, jsonl_path, encoding="utf-8"):
    with open(csv_path, mode="r", encoding=encoding, newline="") as csv_file:
        reader = csv.DictReader(csv_file)

        with open(jsonl_path, mode="w", encoding=encoding) as jsonl_file:
            for row in reader:
                json_line = json.dumps(row, ensure_ascii=False)
                jsonl_file.write(json_line + "\n")



In [ ]:
# Function reading jsonl file

def read_json_data(jsonl_path):
    all_json_data = [] # a list of json records
    with open(jsonl_path, "r", encoding="utf-8") as file:
        for line in file:
            json_data = json.loads(line)
            all_json_data.append(json_data)
    return all_json_data

In [ ]:
# Function Create index.
# Use bulk helpers

from elasticsearch.helpers import bulk

def create_index(json_data_list, index_name):
    documents = []

    for doc in json_data_list:
        documents.append({
            "_index": index_name,
            "_source": doc
        })

    success, errors = bulk(
        client,
        documents,
        refresh=True,
        raise_on_error=False   
    )

    print("Bulk success:", success)
    print("Bulk errors:", len(errors))

    if errors:
        print("First error example:")
        print(errors[0])

    print(f"Done indexing documents into {index_name} index!")


In [ ]:
# Function to delete the index anytime

def delete_index(index_name):
    if client.indices.exists(index=index_name):
        client.indices.delete(index=index_name)
        print(f"Index '{index_name}' deleted.")
    else:
        print(f"Index '{index_name}' does not exists.")

In [ ]:
# Function that prints the text of a query

def search_response(response):
    if len(response["hits"]["hits"]) == 0:
        print("Your search returned no results.")
    else:
        for hit in response["hits"]["hits"]:
            id = hit["_id"]
            score = hit["_score"]
            text = hit["_source"]['Text']
            output = f"id: {id}\nScore: {score}\ntext: {text}\n"

            print(output)

In [ ]:
# Function for creating run txt files for trec_eval

def search_response_trec(response, query_id, run_name, file):
    rank = 1
    for hit in response["hits"]["hits"]:
        doc_id = hit["_source"]["ID"]   # ID από dataset
        score = hit["_score"]

        line = f"{query_id} Q0 {doc_id} {rank} {score} {run_name}\n"
        file.write(line)

        rank += 1


In [ ]:
# Function get_query that extracts all the text for a specific id

def get_query(all_json_queries, search_id):
    for q in all_json_queries:
        if q['ID'] == search_id:
            text = q['Text']
            return text

In [ ]:
# Conversion of the file documents.csv to documents.jsonl

csv_to_jsonl(doc_csv_path, doc_jsonl_path)

In [ ]:
# Conversion of the file queries.csv to queries.jsonl
csv_to_jsonl(queries_csv_path, queries_jsonl_path)

In [14]:
# List of data in documents.jsonl

all_json_data = read_json_data(doc_jsonl_path)

In [ ]:
# List of queries in queries.jsonl

all_json_queries = read_json_data(queries_jsonl_path)
# print(all_json_queries)

for item in all_json_queries:
    print(f"id: {item['ID']}\ntext: {item['Text']}\n")

In [16]:
# Type and length of the data in all_json_data of documents.jsonl

print("Type of all_json_data list:", type(all_json_data))
print("Length of all_json_data list:", len(all_json_data))

Type of all_json_data list: <class 'list'>
Length of all_json_data list: 18316


In [ ]:
# Take a peek of the data in all_json_data of documents.jsonl

for item in all_json_data[0:5]:
    print(f"id: {item['ID']}\ntext: {item['Text']}\n")

In [ ]:
# Call delete_index(index_name) any time - check before call create_index function

index_name = "my_data"
delete_index(index_name)

In [ ]:
# configure index properties
# create fields

mapping = {
  "settings": {
    "number_of_shards": 1,
    "index": {
      "similarity": {
        "default": {
          "type": "BM25",
          "b": 0.8,
          "k1": 1.8
        }
      }
    },
    "analysis": {
      "analyzer": {
        "default": {
          "type": "english"
        },
        "default_search": {
          "type": "english"
        }
      }
    }
  },
  "mappings": {
    "properties": {
      "id": {
        "type": "keyword"
      },
      "text": {
        "type": "text"
      }
    }
  }
}

# create an index with the configuration above
client.indices.create(index='my_data', body=mapping)

In [ ]:
# view mapping

client.indices.get_mapping(index="my_data")

In [ ]:
# Call create_index(json_data_list, index_name)

my_data = create_index(all_json_data, "my_data")


In [ ]:
# Inspect the ElasticSearch index

# Check my_data index if exists
print("'my_data' index exists:", client.indices.exists(index="my_data"))

# Check list of all_json_data
length_of_all_json_data = len(all_json_data)
print("Length of all_json_data list:", length_of_all_json_data)

# count documents in my_data index
# res = client.count(index="my_data")
# print(res["count"])

res = client.count(index="my_data")
print("Number of documents in client index:", res["count"])

In [ ]:
# The following query: match_all essentially prints the entire index.
search_results = client.search(index="my_data", query={"match_all": {}})

# Print search results - call search_response(search_results)
search_response(search_results)

In [ ]:
# full text search - query 1      
search_id = ["Q01", "Q02", "Q03", "Q04", "Q05", "Q06", "Q07", "Q08", "Q09", "Q10"]
k = 20
 
for i in range(len(search_id)):
    text = get_query(all_json_queries, search_id[i])
    search_results = client.search(
        index="my_data",
        size=k,
        query = {"match": {"Text": text}},
    )

    # Print search results
    print(f"Results for k = {k}\nQuery: {search_id[i]}\n")
    search_response(search_results)

In [ ]:
# full text search για k = 20      
search_id = ["Q01", "Q02", "Q03", "Q04", "Q05", "Q06", "Q07", "Q08", "Q09", "Q10"]
k = 20
 
for i in range(len(search_id)):
    text = get_query(all_json_queries, search_id[i])
    search_results = client.search(
        index="my_data",
        size=k,
        query = {"match": {"Text": text}},
    )

    # Print search results
    print(f"{search_id[i]}")
    search_response(search_results)

In [ ]:
# full text search για k = 30     
search_id = ["Q01", "Q02", "Q03", "Q04", "Q05", "Q06", "Q07", "Q08", "Q09", "Q10"]
k = 30
 
for i in range(len(search_id)):
    text = get_query(all_json_queries, search_id[i])
    search_results = client.search(
        index="my_data",
        size=k,
        query = {"match": {"Text": text}},
    )

    # Print search results
    print(f"{search_id[i]}")
    search_response(search_results)

In [ ]:
# full text search για k = 50      
search_id = ["Q01", "Q02", "Q03", "Q04", "Q05", "Q06", "Q07", "Q08", "Q09", "Q10"]
k = 50
 
for i in range(len(search_id)):
    text = get_query(all_json_queries, search_id[i])
    search_results = client.search(
        index="my_data",
        size=k,
        query = {"match": {"Text": text}},
    )

    # Print search results
    print(f"{search_id[i]}")
    search_response(search_results)

In [ ]:
# write file with k=20 for trec_eval

search_ids = ["Q01", "Q02", "Q03", "Q04", "Q05", "Q06", "Q07", "Q08", "Q09", "Q10"]

k = 20
run_name = "my_system"

with open("run_k20.txt", "w", encoding="utf-8") as f:
    for qid in search_ids:
        query_text = get_query(all_json_queries, qid)

        search_results = client.search(
            index="my_data",
            size=k,
            query={"match": {"Text": query_text}},
        )

        search_response_trec(search_results, qid, run_name, f)


In [ ]:
# write file with k=30 for trec_eval

search_ids = ["Q01", "Q02", "Q03", "Q04", "Q05", "Q06", "Q07", "Q08", "Q09", "Q10"]

k = 30
run_name = "my_system"

with open("run_k30.txt", "w", encoding="utf-8") as f:
    for qid in search_ids:
        query_text = get_query(all_json_queries, qid)

        search_results = client.search(
            index="my_data",
            size=k,
            query={"match": {"Text": query_text}},
        )

        search_response_trec(search_results, qid, run_name, f)

In [ ]:
# write file with k=50 for trec_eval

search_ids = ["Q01", "Q02", "Q03", "Q04", "Q05", "Q06", "Q07", "Q08", "Q09", "Q10"]

k = 50
run_name = "my_system"

with open("run_k50.txt", "w", encoding="utf-8") as f:
    for qid in search_ids:
        query_text = get_query(all_json_queries, qid)

        search_results = client.search(
            index="my_data",
            size=k,
            query={"match": {"Text": query_text}},
        )

        search_response_trec(search_results, qid, run_name, f)